# Symmetric Group S_n: Spherical Training with Shared Embedding

This notebook demonstrates Stage 1 training on the **symmetric group S_n** with **shared embedding** (`share_embed=True`).

**Task**: Permutation composition σ ∘ τ (non-commutative)

**Basis**: Irreducible representations of S_n (Young tableaux)

**Note**: With shared embedding, both positions use the same embedding vectors. This may not be ideal for non-commutative operations like permutation composition, but we test it for comparison.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from math import factorial

from src.tasks import SymmetricGroupMultiplication
from src.basis import SymmetricGroupBasis
from src.model import WideNetworkScaleSphere
from src.utils import (
    cross_entropy_high_precision,
    accuracy,
    project_gradient_to_tangent_space,
    normalize_to_pi,
)

# Custom colormap: blue -> white -> red
blue_white_red = LinearSegmentedColormap.from_list(
    'blue_white_red',
    ['#0D2758', 'white', '#A32015'],
    N=256
)

## 1. Configuration

In [2]:
# === Symmetric Group Configuration ===
N_DEGREE = 4  # S_4 has 24 elements (4! = 24)
              # S_3 has 6 elements (3! = 6)
              # S_5 has 120 elements (5! = 120)

# Model
WIDTH = 1024
ACT_TYPE = 'Quad'
INIT_SCALE = 0.5
SHARE_EMBED = True  # Shared embedding for both positions

# Training
LR = 1e-3
NUM_EPOCHS = 2000
PRINT_EVERY = 500

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cpu')

vocab_size = factorial(N_DEGREE)
print(f"Symmetric Group: S_{N_DEGREE}")
print(f"Group size: |S_{N_DEGREE}| = {N_DEGREE}! = {vocab_size}")
print(f"Dataset size: {vocab_size}² = {vocab_size**2} pairs")
print(f"Model: width={WIDTH}, activation={ACT_TYPE}, share_embed={SHARE_EMBED}")

Symmetric Group: S_4
Group size: |S_4| = 4! = 24
Dataset size: 24² = 576 pairs
Model: width=1024, activation=Quad, share_embed=True


## 2. Initialize Task, Basis, and Model

In [3]:
# Create task and basis
task = SymmetricGroupMultiplication(n=N_DEGREE)
basis = SymmetricGroupBasis(n=N_DEGREE)

print(f"Task: {task}")
print(f"Basis: {basis}")
print(f"\nIrreducible representations (partitions -> dimension):")
for partition, dim in basis.partition_dimensions.items():
    part_str = ','.join(map(str, partition))
    print(f"  [{part_str}]: dim = {dim} -> contributes {dim**2} basis functions")
print(f"\nTotal basis functions: {sum(d**2 for d in basis.partition_dimensions.values())} = {vocab_size}")

Task: SymmetricGroupMultiplication(n=4, vocab_size=24)
Basis: SymmetricGroupBasis(n=4, size=24, irreps={[4]:1, [3,1]:3, [2,2]:2, [2,1,1]:3, [1,1,1,1]:1})

Irreducible representations (partitions -> dimension):
  [4]: dim = 1 -> contributes 1 basis functions
  [3,1]: dim = 3 -> contributes 9 basis functions
  [2,2]: dim = 2 -> contributes 4 basis functions
  [2,1,1]: dim = 3 -> contributes 9 basis functions
  [1,1,1,1]: dim = 1 -> contributes 1 basis functions

Total basis functions: 24 = 24


In [4]:
# Generate data
data, labels = task.generate_all_data(device)

# Create model
model = WideNetworkScaleSphere(
    d_vocab=vocab_size,
    width=WIDTH,
    act_type=ACT_TYPE,
    init_scale=INIT_SCALE,
    share_embed=SHARE_EMBED,
).to(device)

print(f"\nModel architecture:")
print(f"  W_in:  {model.W_in.shape}  <- (width, vocab_size) for shared")
print(f"  W_out: {model.W_out.shape}")
print(f"  scale_in:  {model.scale_in.item():.4f}")
print(f"  scale_out: {model.scale_out.item():.4f}")
print(f"\nDataset: {len(data)} samples")


Model architecture:
  W_in:  torch.Size([1024, 24])  <- (width, vocab_size) for shared
  W_out: torch.Size([24, 1024])
  scale_in:  0.5000
  scale_out: 0.5000

Dataset: 576 samples


## 3. Run Stage 1 Training (Projected GD on Sphere)

In [5]:
# Save initial weights before training
W_in_init = model.W_in.detach().cpu().clone()
W_out_init = model.W_out.detach().cpu().clone()

# Freeze scales
model.scale_in.requires_grad = False
model.scale_out.requires_grad = False

# Optimizer for directions only
optimizer = optim.Adam([model.W_in, model.W_out], lr=LR)

# Training history
losses = []
accuracies = []

print("=" * 60)
print("Stage 1: Projected GD (Directions only, Scale fixed)")
print("=" * 60)
print(f"Fixed scale_in = {model.scale_in.item():.4f}, scale_out = {model.scale_out.item():.4f}")

for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    
    # Forward
    logits = model(data)
    loss = cross_entropy_high_precision(logits, labels)
    acc = accuracy(logits, labels)
    
    # Backward
    loss.backward()
    
    # Project gradients onto tangent space
    with torch.no_grad():
        model.W_in.grad = project_gradient_to_tangent_space(
            model.W_in.data, model.W_in.grad, dim=1
        )
        model.W_out.grad = project_gradient_to_tangent_space(
            model.W_out.data, model.W_out.grad, dim=0
        )
    
    # Step
    optimizer.step()
    
    # Project back onto sphere
    model.normalize_directions()
    
    # Record
    losses.append(loss.item())
    accuracies.append(acc)
    
    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch:5d} | Loss: {loss.item():.6f} | Acc: {acc:.4f}")

print(f"\nFinal | Loss: {losses[-1]:.6f} | Acc: {accuracies[-1]:.4f}")

Stage 1: Projected GD (Directions only, Scale fixed)
Fixed scale_in = 0.5000, scale_out = 0.5000
Epoch     0 | Loss: 3.178056 | Acc: 0.0434
Epoch   500 | Loss: 3.176669 | Acc: 0.0521
Epoch  1000 | Loss: 3.176647 | Acc: 0.0503
Epoch  1500 | Loss: 3.176646 | Acc: 0.0503

Final | Loss: 3.176646 | Acc: 0.0503


## 4. Single Frequency Phenomenon (Representation Analysis)

For `share_embed=True`, W_in has shape `(width, vocab_size)` - a single embedding shared by both positions.

The Fourier transform on S_n uses irreducible representations indexed by partitions.

In [6]:
# Get trained weights
W_in = model.W_in.detach().cpu()
W_out = model.W_out.detach().cpu()

# Number of neurons
M = WIDTH

print(f"W_in shape: {W_in.shape}  (shared embedding)")
print(f"W_out shape: {W_out.shape}")

# Normalize (trained)
W_in_norm = W_in / W_in.norm(dim=1, keepdim=True)
W_out_norm = W_out / W_out.norm(dim=0, keepdim=True)

# Normalize (initial)
W_in_init_norm = W_in_init / W_in_init.norm(dim=1, keepdim=True)
W_out_init_norm = W_out_init / W_out_init.norm(dim=0, keepdim=True)

# Apply Fourier transform (trained)
W_in_dft = basis.transform(W_in_norm)
W_out_dft = basis.transform(W_out_norm.T)

# Apply Fourier transform (initial)
W_in_init_dft = basis.transform(W_in_init_norm)
W_out_init_dft = basis.transform(W_out_init_norm.T)

# Get magnitudes (trained)
W_in_mag = torch.abs(W_in_dft).numpy()
W_out_mag = torch.abs(W_out_dft).numpy()

# Get magnitudes (initial)
W_in_init_mag = torch.abs(W_in_init_dft).numpy()
W_out_init_mag = torch.abs(W_out_init_dft).numpy()

print(f"\nDFT shapes:")
print(f"W_in_dft: {W_in_dft.shape}")
print(f"W_out_dft: {W_out_dft.shape}")

# Compute partition boundaries for grouping
partition_boundaries = []
partition_labels = []
curr_idx = 0
for partition in basis.partitions:
    dim = basis.partition_dimensions[partition]
    partition_boundaries.append((curr_idx, curr_idx + dim * dim))
    part_str = ','.join(map(str, partition))
    partition_labels.append(f"[{part_str}]\n(d={dim})")
    curr_idx += dim * dim

print(f"\nPartition blocks: {partition_boundaries}")

W_in shape: torch.Size([1024, 24])  (shared embedding)
W_out shape: torch.Size([24, 1024])

DFT shapes:
W_in_dft: torch.Size([1024, 24])
W_out_dft: torch.Size([1024, 24])

Partition blocks: [(0, 1), (1, 10), (10, 14), (14, 23), (23, 24)]


### 4.1 Single Representation Phenomenon (Representation-Level View)

Unlike cyclic groups where each irrep is 1-dimensional, S_n has irreps of varying dimensions.
The "single frequency" phenomenon here means neurons specialize to a **single irreducible representation** (partition).

We compute the total energy in each irrep block:
$$E_\lambda^{(m)} = \sum_{i,j=1}^{d_\lambda} |\hat{w}_m^{(\lambda, i, j)}|^2$$

In [7]:
# Compute energy per representation for each neuron
def compute_irrep_energy(mag_matrix, boundaries):
    """Compute total squared magnitude in each irrep block."""
    n_neurons = mag_matrix.shape[0]
    n_irreps = len(boundaries)
    energy = np.zeros((n_neurons, n_irreps))
    
    for i, (start, end) in enumerate(boundaries):
        # Sum of squared magnitudes in this block
        energy[:, i] = np.sum(mag_matrix[:, start:end] ** 2, axis=1)
    
    return energy

# Compute energies (trained and init)
energy_in = compute_irrep_energy(W_in_mag, partition_boundaries)
energy_out = compute_irrep_energy(W_out_mag, partition_boundaries)

energy_in_init = compute_irrep_energy(W_in_init_mag, partition_boundaries)
energy_out_init = compute_irrep_energy(W_out_init_mag, partition_boundaries)

print("Energy per irrep (first 8 neurons, W_in):")
print(f"{'Neuron':<8}", end="")
for label in partition_labels:
    print(f"{label.split(chr(10))[0]:<12}", end="")
print()
print("-" * 70)
for m in range(8):
    print(f"{m:<8}", end="")
    for e in energy_in[m]:
        print(f"{e:<12.4f}", end="")
    print()

Energy per irrep (first 8 neurons, W_in):
Neuron  [4]         [3,1]       [2,2]       [2,1,1]     [1,1,1,1]   
----------------------------------------------------------------------
0       0.0000      0.0000      1.0000      0.0000      0.0000      
1       0.0000      1.0000      0.0000      0.0000      0.0000      
2       0.0000      1.0000      0.0000      0.0000      0.0000      
3       0.0000      0.0000      0.0000      1.0000      0.0000      
4       0.0000      0.0000      0.0000      1.0000      0.0000      
5       0.0000      0.0000      0.0000      0.0000      1.0000      
6       0.0000      1.0000      0.0000      0.0000      0.0000      
7       0.0000      0.0000      1.0000      0.0000      0.0000      


In [ ]:
# Plot representation-level energy: Init (gray) vs Trained (colored)
num_neurons = 8
n_irreps = len(partition_boundaries)
x = np.arange(n_irreps)
bar_width = 0.35

fig, axes = plt.subplots(num_neurons, 2, figsize=(10, 1.5 * num_neurons))

for m in range(num_neurons):
    # Find dominant irrep for this neuron
    dom_irrep = np.argmax(energy_in[m])
    
    # W_in (shared)
    ax = axes[m, 0]
    ax.bar(x - bar_width/2, energy_in_init[m], bar_width, alpha=0.6, color='gray', label='Init')
    bars = ax.bar(x + bar_width/2, energy_in[m], bar_width, alpha=0.8, color='#0D2758', label='Trained')
    bars[dom_irrep].set_color('red')
    bars[dom_irrep].set_alpha(1.0)
    ax.set_ylabel(f'Neuron {m}')
    ax.set_xticks(x)
    ax.set_xticklabels([l.split('\n')[0] for l in partition_labels], fontsize=9)
    if m == 0:
        ax.set_title(r'Energy per Irrep: $W_{\mathrm{in}}$ (shared)')
        ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Output
    ax = axes[m, 1]
    ax.bar(x - bar_width/2, energy_out_init[m], bar_width, alpha=0.6, color='gray', label='Init')
    bars = ax.bar(x + bar_width/2, energy_out[m], bar_width, alpha=0.8, color='#A32015', label='Trained')
    bars[dom_irrep].set_color('red')
    bars[dom_irrep].set_alpha(1.0)
    ax.set_xticks(x)
    ax.set_xticklabels([l.split('\n')[0] for l in partition_labels], fontsize=9)
    if m == 0:
        ax.set_title(r'Energy per Irrep: $W_{\mathrm{out}}$')
        ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.suptitle('Single Representation Phenomenon (Shared Embedding)', 
             y=1.01, fontsize=12, fontweight='bold')
plt.show()

### 4.2 Fine-Grained View: Individual Basis Functions

This shows the detailed view at the individual basis function level (ρ_λ(g)_{ij}), with partition boundaries marked by dotted lines.

In [ ]:
# Plot DFT comparison: Init (gray) vs Stage 1 (color)
# One neuron per representation (first neuron by index that specializes in each irrep)
freq_indices = np.arange(vocab_size)

# High threshold since neurons fully specialize to one irrep
ENERGY_THRESHOLD = 0.99

# Find one neuron per irrep
neurons_per_irrep = []
for partition in basis.partitions:
    irrep_idx = list(basis.partitions).index(partition)
    dominant_neurons = np.where(energy_in[:, irrep_idx] > ENERGY_THRESHOLD)[0]
    if len(dominant_neurons) > 0:
        neurons_per_irrep.append((partition, dominant_neurons[0]))
    else:
        part_str = ','.join(map(str, partition))
        print(f"Skipping [{part_str}]: no neurons with >{ENERGY_THRESHOLD*100:.0f}% energy")

num_rows = len(neurons_per_irrep)

# Build basis labels (i,j) for each frequency (1-based indexing)
basis_labels = []
for partition in basis.partitions:
    d = basis.partition_dimensions[partition]
    for i in range(d):
        for j in range(d):
            basis_labels.append(f'({i+1},{j+1})')

# Compute center position for each irrep block (for representation labels)
irrep_centers = []
irrep_labels_short = []
for start, end in partition_boundaries:
    center = (start + end - 1) / 2
    irrep_centers.append(center)
for partition in basis.partitions:
    part_str = ','.join(map(str, partition))
    irrep_labels_short.append(f'[{part_str}]')

fig, axes = plt.subplots(num_rows, 2, figsize=(14, 2.5 * num_rows))
if num_rows == 1:
    axes = axes.reshape(1, -1)

for row, (partition, m) in enumerate(neurons_per_irrep):
    part_str = ','.join(map(str, partition))
    is_last_row = (row == num_rows - 1)
    
    # W_in (shared) - Blue
    ax = axes[row, 0]
    ax.bar(freq_indices, W_in_init_mag[m], alpha=0.4, color='gray', label='Init')
    ax.bar(freq_indices, W_in_mag[m], alpha=0.8, color='#0D2758', label='Trained')
    for start, end in partition_boundaries:
        if start > 0:
            ax.axvline(x=start-0.5, color='black', linestyle='-', linewidth=1.5, alpha=0.8)
    ax.set_ylabel(f'$[{part_str}]$ Neuron {m}', fontsize=9)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    if row == 0:
        ax.set_title(r'$|\hat{\theta}_m|$ ($W_{\mathrm{in}}$ shared)')
        ax.legend(loc='upper right', fontsize=8)
    if is_last_row:
        ax.set_xticks(freq_indices)
        ax.set_xticklabels(basis_labels, fontsize=7, rotation=90)
        for i, (center, label) in enumerate(zip(irrep_centers, irrep_labels_short)):
            ax.annotate(f'${label}$', xy=(center, -0.25), xycoords=('data', 'axes fraction'),
                       ha='center', va='top', fontsize=9)
    else:
        ax.set_xticks([])
    
    # Output (ξ_m) - Red
    ax = axes[row, 1]
    ax.bar(freq_indices, W_out_init_mag[m], alpha=0.4, color='gray', label='Init')
    ax.bar(freq_indices, W_out_mag[m], alpha=0.8, color='#A32015', label='Trained')
    for start, end in partition_boundaries:
        if start > 0:
            ax.axvline(x=start-0.5, color='black', linestyle='-', linewidth=1.5, alpha=0.8)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    if row == 0:
        ax.set_title(r'$|\hat{\xi}_m|$ ($W_{\mathrm{out}}$)')
        ax.legend(loc='upper right', fontsize=8)
    if is_last_row:
        ax.set_xticks(freq_indices)
        ax.set_xticklabels(basis_labels, fontsize=7, rotation=90)
        for i, (center, label) in enumerate(zip(irrep_centers, irrep_labels_short)):
            ax.annotate(f'${label}$', xy=(center, -0.25), xycoords=('data', 'axes fraction'),
                       ha='center', va='top', fontsize=9)
    else:
        ax.set_xticks([])

plt.tight_layout()
plt.subplots_adjust(bottom=0.12)
plt.show()

## 5. Phase Alignment via Matrix Product

For **shared embedding**, both positions use the same embedding $\hat{W}_{in}$.

The phase alignment condition becomes:
$$\hat{W}_{in}(\lambda) \cdot \hat{W}_{in}(\lambda) \cdot \hat{W}_{out}(\lambda)^\dagger = \hat{W}_{in}^2(\lambda) \cdot \hat{W}_{out}(\lambda)^\dagger \approx \text{real matrix}$$

where each $\hat{W}(\lambda)$ is a $d_\lambda \times d_\lambda$ matrix.

In [10]:
# Extract matrix-valued Fourier coefficients for each irrep
def extract_irrep_matrices(dft_coeffs, partitions, partition_dims):
    """
    Extract d_λ × d_λ matrices from flattened DFT coefficients.
    
    Args:
        dft_coeffs: shape (n_neurons, |G|) - flattened Fourier coefficients
        partitions: list of partitions
        partition_dims: dict mapping partition -> dimension
    
    Returns:
        dict mapping partition -> array of shape (n_neurons, d_λ, d_λ)
    """
    n_neurons = dft_coeffs.shape[0]
    irrep_matrices = {}
    
    curr_idx = 0
    for partition in partitions:
        d = partition_dims[partition]
        # Extract d² coefficients and reshape to (n_neurons, d, d)
        block = dft_coeffs[:, curr_idx:curr_idx + d*d]
        irrep_matrices[partition] = block.reshape(n_neurons, d, d)
        curr_idx += d * d
    
    return irrep_matrices

# Extract matrices for W_in and W_out
W_in_matrices = extract_irrep_matrices(W_in_dft.numpy(), basis.partitions, basis.partition_dimensions)
W_out_matrices = extract_irrep_matrices(W_out_dft.numpy(), basis.partitions, basis.partition_dimensions)

print("Extracted irrep matrices:")
for partition, mat in W_in_matrices.items():
    d = basis.partition_dimensions[partition]
    part_str = ','.join(map(str, partition))
    print(f"  [{part_str}]: shape = {mat.shape} (neurons × {d} × {d})")

Extracted irrep matrices:
  [4]: shape = (1024, 1, 1) (neurons × 1 × 1)
  [3,1]: shape = (1024, 3, 3) (neurons × 3 × 3)
  [2,2]: shape = (1024, 2, 2) (neurons × 2 × 2)
  [2,1,1]: shape = (1024, 3, 3) (neurons × 3 × 3)
  [1,1,1,1]: shape = (1024, 1, 1) (neurons × 1 × 1)


In [11]:
# Compute the matrix product: W_in @ W_in @ W_out^† = W_in² @ W_out^†
# For shared embedding, both positions use the same W_in

def compute_phase_product_shared(W_in, Wout):
    """
    Compute W_in @ W_in @ Wout^† = W_in² @ Wout^† for each neuron.
    
    Args:
        W_in, Wout: arrays of shape (n_neurons, d, d)
    
    Returns:
        product: array of shape (n_neurons, d, d)
    """
    n_neurons = W_in.shape[0]
    d = W_in.shape[1]
    product = np.zeros((n_neurons, d, d), dtype=complex)
    
    for m in range(n_neurons):
        product[m] = W_in[m] @ W_in[m] @ Wout[m].conj().T
    
    return product

def compute_realness_ratio(product):
    """Compute ||Im||_F / ||total||_F for each neuron's matrix."""
    imag_norms = np.array([np.linalg.norm(product[m].imag, 'fro') for m in range(product.shape[0])])
    total_norms = np.array([np.linalg.norm(product[m], 'fro') for m in range(product.shape[0])])
    return imag_norms / (total_norms + 1e-10)

# High threshold since neurons fully specialize to one irrep
ENERGY_THRESHOLD = 0.99

# Check phase alignment for each irrep
print("Phase alignment check: Is W_in² @ W_out^† approximately real?")
print(f"(Only neurons with >{ENERGY_THRESHOLD*100:.0f}% energy in their dominant irrep)")
print("=" * 70)
print(f"{'Irrep':<12} {'# Neurons':<12} {'Mean |Im|/|tot|':<20} {'Real?':<15}")
print("-" * 70)

all_ratios = []

for partition in basis.partitions:
    d = basis.partition_dimensions[partition]
    part_str = ','.join(map(str, partition))
    
    W_in_mat = W_in_matrices[partition]
    Wout = W_out_matrices[partition]
    
    # Compute W_in² @ Wout^†
    product = compute_phase_product_shared(W_in_mat, Wout)
    
    # Only consider neurons with high energy in this irrep
    irrep_idx = list(basis.partitions).index(partition)
    dominant_neurons = np.where(energy_in[:, irrep_idx] > ENERGY_THRESHOLD)[0]
    
    if len(dominant_neurons) > 0:
        ratios = compute_realness_ratio(product[dominant_neurons])
        mean_ratio = np.mean(ratios)
        all_ratios.extend(ratios)
        
        is_real = "YES" if mean_ratio < 0.1 else "NO"
        print(f"[{part_str}]{'':>{10-len(part_str)}} {len(dominant_neurons):<12} {mean_ratio:<20.4f} {is_real:<15}")
    else:
        print(f"[{part_str}]{'':>{10-len(part_str)}} {'0':<12} {'N/A':<20} {'(skipped)':<15}")

print("-" * 70)
if len(all_ratios) > 0:
    print(f"\nOverall: Mean |Im|/|total| ratio = {np.mean(all_ratios):.4f}")
    print(f"Phase alignment {'SATISFIED' if np.mean(all_ratios) < 0.1 else 'NOT SATISFIED'} (threshold: 0.1)")

Phase alignment check: Is W_in² @ W_out^† approximately real?
(Only neurons with >99% energy in their dominant irrep)
Irrep        # Neurons    Mean |Im|/|tot|      Real?          
----------------------------------------------------------------------
[4]          0            N/A                  (skipped)      
[3,1]        310          0.0000               YES            
[2,2]        209          0.0000               YES            
[2,1,1]      327          0.0000               YES            
[1,1,1,1]    178          0.0000               YES            
----------------------------------------------------------------------

Overall: Mean |Im|/|total| ratio = 0.0000
Phase alignment SATISFIED (threshold: 0.1)


In [ ]:
# Visualize the matrix products for selected neurons per irrep
# For each irrep: show Real and Imaginary parts for the first neuron (by index) in that irrep
# Skip irreps with no dominant neurons

# High threshold since neurons fully specialize to one irrep
ENERGY_THRESHOLD = 0.99

# Find which irreps have dominant neurons
irreps_with_neurons = []
for partition in basis.partitions:
    irrep_idx = list(basis.partitions).index(partition)
    dominant_neurons = np.where(energy_in[:, irrep_idx] > ENERGY_THRESHOLD)[0]
    if len(dominant_neurons) > 0:
        irreps_with_neurons.append((partition, dominant_neurons[0]))  # (partition, first neuron by index)
    else:
        part_str = ','.join(map(str, partition))
        print(f"Skipping [{part_str}]: no neurons with >{ENERGY_THRESHOLD*100:.0f}% energy in this irrep")

if len(irreps_with_neurons) > 0:
    fig, axes = plt.subplots(len(irreps_with_neurons), 3, figsize=(5, 2 * len(irreps_with_neurons)),
                             gridspec_kw={'width_ratios': [1, 1, 0.08], 
                                          'wspace': 0.02, 'hspace': 0.15})
    if len(irreps_with_neurons) == 1:
        axes = axes.reshape(1, -1)

    for row, (partition, m) in enumerate(irreps_with_neurons):
        d = basis.partition_dimensions[partition]
        part_str = ','.join(map(str, partition))
        
        W_in_mat = W_in_matrices[partition]
        Wout = W_out_matrices[partition]
        
        product = compute_phase_product_shared(W_in_mat, Wout)
        
        # Real part
        ax = axes[row, 0]
        im = ax.imshow(product[m].real, cmap=blue_white_red, vmin=-1, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_ylabel(f'$[{part_str}]$ Neuron {m}', fontsize=9)
        if row == 0:
            ax.set_title(r'Re$(\hat{W}_{\mathrm{in}}^2 \hat{W}_{\mathrm{out}}^\dagger)$', fontsize=9)
        
        # Imaginary part
        ax = axes[row, 1]
        im = ax.imshow(product[m].imag, cmap=blue_white_red, vmin=-1, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(r'Im$(\hat{W}_{\mathrm{in}}^2 \hat{W}_{\mathrm{out}}^\dagger)$', fontsize=9)
        
        # Colorbar for this row
        cbar = fig.colorbar(im, cax=axes[row, 2])
        cbar.set_ticks([-1, 0, 1])

    plt.show()

In [13]:
# Summary: For neurons with single-irrep specialization
# Metric: Frobenius norm of imaginary part / Frobenius norm of full matrix

# High threshold since neurons fully specialize to one irrep
ENERGY_THRESHOLD = 0.99

# Find neurons that fully specialize to one irrep
specialized_neurons = []
for m in range(WIDTH):
    max_energy = np.max(energy_in[m])
    if max_energy > ENERGY_THRESHOLD:
        specialized_neurons.append(m)

print(f"Summary: Phase alignment for neurons with >{ENERGY_THRESHOLD*100:.0f}% energy in one irrep")
print(f"Found {len(specialized_neurons)} specialized neurons out of {WIDTH}")
print("=" * 80)
print(f"{'Neuron':<8} {'Irrep':<12} {'|Im|/|tot| Ratio':<18} {'Real?':<8}")
print("-" * 80)

ratios = []

for m in specialized_neurons[:20]:  # Show first 20 specialized neurons
    # Find dominant irrep for this neuron
    dom_irrep_idx = np.argmax(energy_in[m])
    partition = list(basis.partitions)[dom_irrep_idx]
    part_str = ','.join(map(str, partition))
    
    W_in_mat = W_in_matrices[partition][m]
    Wout = W_out_matrices[partition][m]
    
    # W_in² @ Wout†
    product = W_in_mat @ W_in_mat @ Wout.conj().T
    ratio = np.linalg.norm(product.imag, 'fro') / (np.linalg.norm(product, 'fro') + 1e-10)
    ratios.append(ratio)
    
    is_real = "YES" if ratio < 0.1 else "NO"
    print(f"{m:<8} [{part_str}]{'':>{10-len(part_str)}} {ratio:<18.4f} {is_real:<8}")

print("-" * 80)
print(f"\nOverall statistics (first {len(ratios)} specialized neurons):")
print(f"  Mean ratio: {np.mean(ratios):.4f}")
print(f"  Neurons with ratio < 0.1 (approximately real): {sum(r < 0.1 for r in ratios)}/{len(ratios)}")

Summary: Phase alignment for neurons with >99% energy in one irrep
Found 1024 specialized neurons out of 1024
Neuron   Irrep        |Im|/|tot| Ratio   Real?   
--------------------------------------------------------------------------------
0        [2,2]        0.0000             YES     
1        [3,1]        0.0000             YES     
2        [3,1]        0.0000             YES     
3        [2,1,1]      0.0000             YES     
4        [2,1,1]      0.0000             YES     
5        [1,1,1,1]    0.0000             YES     
6        [3,1]        0.0000             YES     
7        [2,2]        0.0000             YES     
8        [2,2]        0.0000             YES     
9        [3,1]        0.0000             YES     
10       [2,1,1]      0.0000             YES     
11       [2,1,1]      0.0000             YES     
12       [2,1,1]      0.0000             YES     
13       [2,1,1]      0.0000             YES     
14       [3,1]        0.0000             YES     
15       

## 6. Representation Distribution Analysis

Which irreducible representations do neurons specialize to?

In [14]:
# Count which partition each neuron specializes to
partition_counts = {p: 0 for p in basis.partitions}

for m in range(M):
    dom_freq = basis.get_dominant_frequency(W_in_dft[m])
    
    # Find which partition this frequency belongs to
    curr_idx = 0
    for partition in basis.partitions:
        dim = basis.partition_dimensions[partition]
        if dom_freq < curr_idx + dim * dim:
            partition_counts[partition] += 1
            break
        curr_idx += dim * dim

print("Neuron specialization by irreducible representation:")
print("=" * 50)
for partition, count in partition_counts.items():
    dim = basis.partition_dimensions[partition]
    part_str = ','.join(map(str, partition))
    print(f"  [{part_str}] (dim={dim}): {count} neurons ({100*count/M:.1f}%)")

Neuron specialization by irreducible representation:
  [4] (dim=1): 0 neurons (0.0%)
  [3,1] (dim=3): 310 neurons (30.3%)
  [2,2] (dim=2): 209 neurons (20.4%)
  [2,1,1] (dim=3): 327 neurons (31.9%)
  [1,1,1,1] (dim=1): 178 neurons (17.4%)
